# Lesson 6: Raster Data with Rasterio and Numpy

In this lesson, we will explore how to work with raster data using the rasterio and numpy Python libraries. 

Raster data is one of the most common ways to represent spatial information in Earth science, and many satellite products, digital elevation models (DEMs), land cover maps, and other derived datasets are stored as rasters. Rasterio is designed for reading, writing, and manipulating raster datasets. It is built on top of the geospatial library GDAL (Geospatial Data Abstraction Library), which is also the engine behind the QGIS interactive GIS mapping software. 

Some of the concepts in this lesson overlap with concepts covered in the Gridded Data lesson, which is natural given that we can think of raster data as a specific sub group of gridded data. At the end of the lesson, there are instructions on how we could perform the same operations in this lesson using tools from the xarray and rioxarray libraries. As we will discuss in this lesson, which tools you use is very much down to personal preference. However, rioxarray essentially uses rasterio methods behind the scenes and combines them with the xarray data structures. Furthermore, there are specific occasions where rasterio tools are better equipped to perform certain operations (for example, mosaicing many individual `tif` files into a single raster and reprojecting to a new CRS can often be many times more efficient using rasterio versus xarray). For these reasons, it is beneficial for us to explicitly cover rasterio. You should think of rasterio and xarray as two complementary tools at your disposal! 

### Learning outcomes

After this lesson, you should be able to:

1) Define relative and absolute filepaths to data files on your computer.
2) Evaluate the contents of a raster data file using its metadata.
3) Select raster bands and perform raster math using numpy arrays.
4) Create a true color satellite image using a multi-spectral raster data file.
5) Recall the syntax to use numpy.where to mask raster pixels.

:::{danger}
## Entry ticket!
Before we start, discuss with your neighbor the concept you found the most challenging from the last lesson.
:::

## Introduction to Raster Data and Rasterio

In our GeoPandas lesson, we worked with **vector data**, where we represented geographic features using shapely geometry objects such as points, lines, and polygons. Instead of storing individual geometry features, **raster data** represents space as a regular grid of cells (pixels), where each pixel contains a value describing some variable at that location. Raster data is commonly used for satellite imagery, digital elevation models (DEMs), land cover maps, precipitation datasets, and many other Earth science products.

The Python library **Rasterio** is one of the most widely used tools for working with raster datasets. Rasterio provides a simple interface for reading, writing, visualizing, and processing geospatial raster formats such as GeoTIFFs. It builds on the powerful GDAL library while providing a more Python-friendly workflow.

### When to use Rasterio versus Xarray

Both Rasterio and Xarray can be used to work with gridded spatial data, but they serve slightly different purposes. Which library you use is very much a personal choice. 

* I tend to prefer **rasterio** when reading raster data (especially GeoTIFFs), inspecting metadata, clipping rasters, reprojecting datasets, complex merging or resampling, and writing outputs.
* I'll use **xarray** when working with multidimensional datasets (particularly those stored in NetCDF or Zarr formats), and especially when I want to analyze or visualize data or trends over additional dimensions such as time, depth, or ensemble members.

In practice, rasterio and xarray can be great used in combination with one another, and there are additional libraries that make this interaction even easier (see `rioxarray`: https://corteva.github.io/rioxarray/html/getting_started/getting_started.html)

## Harmonized Landsat Sentinel-2 example

Let's open some raster data! The dataset we will work with contains multispectral satellite imagery from Harmonized Landsat Sentinel-2 (HLS). With multi-spectral data, each band contains surface reflectance values for a discrete wavelength window. You can read about the spectral bands provided in HLS data product here: https://www.earthdata.nasa.gov/data/catalog/lpcloud-hlsl30-2.0#variables 

:::{admonition}
## Exercise: Where is my raster data file?

Before we can read the HLS raster data into Python, we need to upload our data to the area we are working in, locate it, and provide the filepath. Since we've already covered the importance of filepaths and the differences between absolute and relative filepaths, let's practice these steps this time:

1. Upload your `.tif` raster file to the working location where you want it.
    
2. Find the relative file path.

3. Find the equivalent absolute file path.

:::

## Reading raster data
Now that we've uploaded our data and defined the file path, let's read the data into Python using rasterio. The first thing we will do is check out the metadata, which will help us understand what this raster data represents and confirm this is what we expected. 

The raster `profile` contains the minimum metadata that would be needed to create another raster with the same structure. We can get additional attributes (`crs`, `bounds`, etc.) from the open raster dataset itself.

In [ ]:
# Import the rasterio library
import rasterio as rio

In [ ]:
# Define the filepath to the HLS raster data 
# Update using the filepath you defined in the last exercise
filepath = './data/hls_20190531_4326.tif'

In [ ]:
# Open the raster file and read its metadata
with rio.open(filepath) as dataset: # Now we have the raster dataset opened
    
    # Read the raster metadata profile - this contains essential info that would enable us to reproduce the raster
    profile = dataset.profile

    # Some other derived attributes we can get from the open raster data file itself
    bounds = dataset.bounds
    res = dataset.res

In [ ]:
# Show the metadata profile
# Notice what kind of data structure this is?
profile

In [ ]:
profile['transform']

## Raster metadata

The actual values in a raster are only part of the dataset. To interpret a raster correctly, we also need **metadata**: information that describes how the raster is stored and how it relates to the real world. Metadata tells us where the raster is located on Earth, what coordinate system it uses, how large the pixels are, and how many bands it contains. Without metadata, a raster is simply a grid of numbers with no geographic context.

| Attribute | Meaning | Example |
|----------|----------|----------|
| `crs` | Coordinate Reference System (CRS) that defines how coordinates relate to locations on Earth | `EPSG:4326` |
| `bounds` | Spatial extent (minimum and maximum coordinates) | `(-120, 35, -118, 37)` |
| `transform` | Affine transformation that maps pixel positions to geographic coordinates | Used by rasterio to convert between pixel locations (row, column) and real-world coordinates |
| `width`, `height` | Number of columns and rows | `1000 × 500` |
| `count` | Number of bands | `3` for an RGB image |
| `res` | Pixel size (spatial resolution) in the units of the CRS | `(10, 10)` meters |

In [ ]:
# Get the raster file format ("driver") 
print(profile['driver'])

# Get the data type of the raster values
print(profile['dtype'])

# Get the EPSG code of the coordinate reference system 
print(profile['crs'].to_epsg())

:::{admonition}
## Exercise: Inspecting raster metadata

Using the metadata `profile` and derived attributes we extracted, can you find the following information about our raster satellite imagery?

1. How many spectral bands does this raster data contain?

2. What are the height and width of the raster data? Bonus: how many total pixels are in this raster data? (Hint: you will need to combine your answers from 1 and 2.)

3. What is the pixel spatial resolution of this data?

4. Where in the world is this data from? (Hint: find the bounds, then use google (or any mapping software) to find roughly where our raster image was captured.) Bonus: can you create a GeoPandas GeoDataFrame containing the raster bounding box? Recall: `gdf = gpd.GeoDataFrame(geometry=[box(xmin, ymin, xmax, ymax)], crs=crs)`

:::

Now let's read in the raster values.

In [ ]:
image = dataset.read() # Read all the bands of raster values into a variable called image

:::{warning}
## Bug alert!

`RasterioIOError: Dataset is closed: ./data/hls_20190531_4326.tif`

What happened? Earlier we used `with rio.open(filepath) as dataset:` to open the raster and extract the metadata. The dataset is no longer open for us to read, and so the computer can't access what's inside. We need to open it again before we can extract the raster values.

:::

In [ ]:
# Open the raster file and read its band values
with rio.open(filepath) as dataset:
    image = dataset.read() # Read all the bands of raster values into a variable called image

In [ ]:
# Shape and size attributes
print(image.shape) # number of bands, number of rows (height), number of columns (width)
print(image.size) # total number of pixels

In [ ]:
image

## Selecting raster bands using indexing

Using rasterio to read in values from a raster dataset returns what's called a numpy array. Numpy is an extremely commonly used Python library providing a multidimensional array data structure and a comprehensive collection of math functions to operate on these arrays. Note that a numpy array also forms the basis (behind the scenes) of the DataArray data structures we've seen using the xarray library. 

Indexing a numpy array uses the `[]` syntax, in which we can perform slicing using `:`. With multi-spectral data, we can use indexing to select certain bands from our raster data. The bands are usually contained in the first dimension after we have read in the raster data from a `.tif` file (although it is always good to check!). Here are the HLS bands in our raster dataset and their index positions: 

| Band Name | Wavelength (nm) | Description | Integer Index |
|:------------|:-------------|:-------------|:---------------|
| B1 | 430-450 | Coastal/Aerosol | 0 |
| B2 | 450-510 | Blue | 1 |
| B3 | 530-590 | Green | 2 |
| B4 | 640-670 | Red | 3 |
| B5 | 850-880| Near Infrared (NIR) | 4 |
| B6 | 1570-1650 | Shortwave Infrared 1 (SWIR1) | 5 |
| B7 | 2110-2290 | Shortwave Infrared 2 (SWIR2) | 6 |

We can also use indexing to select different spatial areas in our data using the numpy array dimensions representing the `x` and `y` coordinates. These are usually the second and third dimensions (but again, it is always good to check first).

## True-color image

Let's create a true-color image of our HLS raster data. A true color image simply uses the red, green, and blue to display images in the same way we naturally see with our eyes.

The figure below illustrates what we call a false-color image, where we change the wavelengths used in each of the RGB bands displayed to help draw out certain interesting features.

![bands_rgb](images/bands_rgb.jpg)

_[Figure from Cal Poly Humboldt Geospatial Curriculum](https://gsp.humboldt.edu/olm/Courses/GSP_216/lessons/composites.html)_

In [ ]:
# Get the red, green, and blue bands from the numpy array
print(image.shape) # 7 bands

red = image[3, :, :]
green = image[2, :, :]
blue = image[1, :, :]

# Now we have two dimensional arrays with dimensions x, y
print(red.shape)
print(green.shape)
print(blue.shape)

In [ ]:
# We can also do the same thing and get the bands during the process of opening and reading the raster file
# BUT! Note how rio read() uses band indexes starting from 1 (not 0!)
with rio.open(filepath) as dataset:
    red = dataset.read(4)
    green = dataset.read(3)
    blue = dataset.read(2)

# Now we have two dimensional arrays with dimensions x, y
print(red.shape)
print(green.shape)
print(blue.shape)

In [ ]:
# Combine red, green, and blue into a single 3D array for plotting RGB image
# We'll use numpy dstack which stacks the bands in the third dimension (which the imshow plotting function expects)
# https://numpy.org/doc/stable/reference/generated/numpy.dstack.html

import numpy as np
import matplotlib.pyplot as plt # for plotting

# Create RGB by stacking
rgb = np.dstack([red, green, blue]) # np.dstack() expects a list of np arrays
print(rgb.shape) # (rows, cols, bands)

# Stretch the contrast of the values and clip them to between 0 and 1
sr_max = 0.15 # Because most of our reflectance values are below 0.15 - feel free to play with this value
rgb = rgb / sr_max # linear stretch
rgb = np.clip(rgb, 0, 1) # clip to between 0 and 1, np.clip(array, min, max)

# Plot using pyplot imshow for raster RGB
plt.imshow(rgb)

In [ ]:
# Indexing / slicing by x, y coordinates to select a smaller area
# Notehow positions are lost
# Feel free to play around with the index slices to change the area
rgb_smallarea = rgb[100:200, 400:600, :] #row, col, band
plt.imshow(rgb_smallarea)

## Raster band math example: Modified Normalized Difference Water Index

Let's use our multi-spectral bands to create an index that helps to detect areas of open water. The **Modified Normalized Difference Water Index (MNDWI)** is commonly used to identify open water in satellite imagery. It is calculated as:

```MNDWI = (Green - SWIR) / (Green + SWIR)```

The index works because water reflects light differently than most land surfaces:

- Water typically reflects moderately in the green wavelengths.
- Water strongly absorbs SWIR radiation, resulting in very low SWIR reflectance.
- Vegetation, soil, and built-up surfaces generally have much higher SWIR reflectance than water.
- Because MNDWI is a normalized difference index, values range between -1 and 1 and are less sensitive to differences in illumination or overall brightness than using individual bands alone.

:::{admonition}
### Exercise: Calculating and plotting Modified Normalized Difference Water Index

Follow these steps to calculate and plot MNDWI for our HLS data:

* Get an array containing the green band. Now, get an array containing the SWIR1 band. (Hint: use the table from earlier to get the integer indexes for these bands.) 
* Perform raster band math to get MNDWI using the formula above.
* Plot the MNDWI image using `plt.imshow()`. Include the arguments `vmin=-1` and `vmax=1`. You can play around with changing the colormap you use (e.g., `cmap='Blues'`) https://matplotlib.org/stable/users/explain/colors/colormaps.html.
* Add a colorbar using `plt.colorbar()`. 

:::

:::{dropdown}

### Solution


```
# Start by getting the green and SWIR bands
green = image[2, :, :]
swir = image[5, :, :]

# Perform the raster band math calculation
mndwi = (green - swir) / (green + swir)

# Plot our water index
plt.imshow(mndwi, vmin=-1, vmax=1, cmap='Blues')
plt.colorbar() # add colorbar
```

:::

## Masking rasters

To wrap up, we'll now mask our MNDWI output to give us a map approximately delineating areas that have highest likelihood of being water. Masking allows us to remove (or hide) areas that meet or fail to meet certain criteria in raster datasets. We set masked areas to some "no data" value, often `np.nan` ("not-a-number") but other values are often also used, depending on the values present in the data and their data types (e.g., 255, -9999, 27682). This is another important component of our metadata we need to keep track of in `profile['nodata']`. For example, cloud masking (removing cloud pixels from optical imagery) is a common step in satellite data processing workflows. 

There are a number of ways to perform masking operations, including masking areas that intersect a vector geometry (read the docs: https://rasterio.readthedocs.io/en/latest/topics/masks.html). Here, we will use a neat numpy function `np.where()` to mask out areas that are not water based on their MNDWI value.

In [ ]:
# Mask out raster pixels with values less than -0.2 MNDWI (not a scientific threshold, just picked visually)
# Feel free to play around with the threshold value (-0.2) and see how the output changes

# np.where(condition, value where true, values where false)
mndwi_masked = np.where(mndwi > -0.2, 1, np.nan)

# Plot the areas we think are most likely to be water
plt.imshow(mndwi_masked)

## Saving raster data to tif file

Having performed some raster processing and operations, we'll often want to save the work we have done. There are lots of options when it comes to saving raster data using rasterio, many of which are related to storing the necessary metadata that would allow someone else to later open and work with our raster data. In our example, we can use the metadata from the original raster file when saving our MNDWI output. The output `GeoTIFF` will retain the original CRS, spatial extent, resolution, and affine transform. 

This is a complete example of all the saving options taken from the [rasterio docs](https://rasterio.readthedocs.io/en/latest/quickstart.html#saving-raster-data). Fortunately, by using the original metadata `profile` we don't need to manually define all of these. But it is good to be aware of what each of these metadata components means and the role they play in relating our data to the real world so that it is not simply a grid of numbers without geographic context.

```
with rio.open(
    '/data/new.tif', #filepath
    'w', #mode, write
    driver='GTiff', # file format
    height=Z.shape[0],
    width=Z.shape[1],
    count=1, # number of bands
    dtype=Z.dtype, # data type: can be a huge factor in the size of the output file
    crs='+proj=latlong',
    transform=transform,
) as dst:
    dst.write(output_image, 1) #filepath, number of raster bands
```

In [ ]:
# Define the file path to save to
output_filepath = './data/hls_20190531_MNDWI.tif'

# Update metadata for the MNDWI output using our original metadata
profile.update(dtype="float32", count=1) # float datatype, one output band

# Write MNDWI to a new GeoTIFF
with rio.open(output_filepath, "w", **profile) as raster_output:
    raster_output.write(mndwi.astype("float32"), 1)

:::{danger}
## Exit ticket!

Before you leave this lesson, please [submit your responses here](https://forms.gle/cVjyNwBZyUycm6RT6) to three quick questions: 

* How much of this lesson's content do you feel comfortable with? 0-10, with 10 being all of the content.
* How was the pace of this lesson for you? 1) Too slow; 2) About right; 3) Too fast.
* In a few words or less, what single concept was most challenging for you in this lesson?

:::

:::{admonition}
### Bonus exercise: False color composite

Earth scientists will often change the wavelengths used to display remotely sensed images to help draw out certain interesting features. These images are known as false color composites, and may use many combinations of bands in the red-green-blue display channels. A common setup for highlighting areas with vegeation uses the NIR, Red, and Green bands for the RGB channels.

Can you create a false color composite image for our HLS raster data? You will need to repeat the true color RGB image we made earlier, this time using the NIR, Red, and Green bands.

:::

:::{hint}
## Bonus concept: Using rioxarray to perform rasterio on xarray DataArrays

Everything we did in this lesson we also could have been performed using the xarray and rioxarray libraries. It's a personal choice whether you feel more comfortable wokring with the DataArray structures provided by xarray or the more fundamental numpy arrays that rasterio uses (although as mentioned at the start of the lesson, there are certain specific operations where one library may make things easier for us). In this simple example, both options seem to work equally well. 

Here's how we could have done it using xarray and rioxarray.
:::

In [ ]:
import xarray as xr
import rioxarray as riox
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------
# 1) Open raster with rioxarray

# Data are loaded as an xarray DataArray with dimensions:
# (band, y, x)
da = riox.open_rasterio(filepath)

# Metadata
print(da.rio.crs)
print(da.rio.bounds())
print(da.rio.resolution())

# Shape information
print(da.shape)  # (bands, rows, cols)
print(da.size)

# ------------------------------------
# 2) Select bands

# Unlike rasterio.read(4), xarray uses coordinate labels
# Here band numbers come directly from the raster (typically 1-based)
red = da.sel(band=4)
green = da.sel(band=3)
blue = da.sel(band=2)

print(red.shape)
print(green.shape)
print(blue.shape)

# ------------------------------------
# 3) Create RGB image for plotting

# Get RGB bands in a new data array rgb
rgb = da.sel(band=[4,3,2])

# Contrast stretch
sr_max = 0.15
rgb = rgb / sr_max
rgb = rgb.clip(0, 1)

# Plot using imshow
rgb.plot.imshow()

# ------------------------------------
# 4) Slice a smaller area

# rgb_smallarea = rgb[100:200, 400:600, :]
rgb_smallarea = rgb.isel(y=slice(100, 200), x=slice(400, 600))
rgb_smallarea.plot.imshow()

# ------------------------------------
# 5) MNDWI calculation

# Select green and SWIR bands
green = da.sel(band=3)
swir = da.sel(band=6)

# Band math works directly on DataArrays
mndwi = (green - swir) / (green + swir)

# Plot with xarray
mndwi.plot(vmin=-1, vmax=1, cmap="Blues")

# ------------------------------------
# 6) Threshold water mask

# xarray equivalent of np.where()
mndwi_masked = xr.where(mndwi > -0.2, 1, np.nan)
mndwi_masked.plot()

# ------------------------------------
# 7) Writing to geotiff
output_filepath = "./data/hls_20190531_MNDWI_xr.tif"

# rioxarray preserves CRS, transform, and metadata automatically
mndwi.attrs["long_name"] = "MNDWI"
mndwi.astype("float32").rio.to_raster(output_filepath)

:::{hint}
## Further reading

* [Beginner's guide to geospatial raster data](https://towardsdatascience.com/the-ultimate-beginners-guide-to-geospatial-raster-data-feb7673f6db0/)
* [Rasterio notebook from GISHub](https://geog-312.gishub.org/book/geospatial/rasterio.html)

Rasterio docs:
* [Plotting and show_hist](https://rasterio.readthedocs.io/en/stable/topics/plotting.html#plotting)
* [Reprojecion](https://rasterio.readthedocs.io/en/stable/topics/reproject.html)
* [Resampling](https://rasterio.readthedocs.io/en/stable/topics/resampling.html)
* [Masks](https://rasterio.readthedocs.io/en/stable/topics/masks.html)
:::